# Crime Data Analysis (2020-Present)

**Category:** 02-Data-Exploration / CSV Datasources  
**Dataset:** Crime_Data_from_2020_to_Present.csv (244MB)  
**Source:** LA Open Data  

---

## Dataset Overview
- Incident-level crime reports from 2020 to present
- Contains location, time, crime type, victim demographics
- Ideal for: Spatiotemporal analysis, pattern detection, hotspot mapping

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy matplotlib seaborn folium

CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
MODEL = CLAUDE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
DATA_PATH = Path.home() / 'lab' / 'data' / 'CSVs' / 'Crime_Data_from_2020_to_Present.csv'

## 1. Load and Explore Data

In [ ]:
# Load dataset (may take a moment - 244MB)
print("Loading crime data...")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} crime records")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Basic info
df.info()

In [ ]:
# Sample records
df.head()

In [ ]:
%%ai $MODEL
Looking at these crime data columns, what are the most important fields for:
1. Temporal analysis (when do crimes occur?)
2. Spatial analysis (where do crimes occur?)
3. Crime type analysis (what types of crimes?)
4. Victim demographics

Also suggest 3 interesting analytical questions we could answer with this data.

## 2. Data Cleaning & Preparation

In [ ]:
# Parse dates
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')

# Extract temporal features
df['Year'] = df['DATE OCC'].dt.year
df['Month'] = df['DATE OCC'].dt.month
df['DayOfWeek'] = df['DATE OCC'].dt.dayofweek
df['Hour'] = df['TIME OCC'] // 100  # Time is in HHMM format

print(f"Date range: {df['DATE OCC'].min()} to {df['DATE OCC'].max()}")

In [ ]:
# Check for missing values in key columns
key_cols = ['DATE OCC', 'TIME OCC', 'AREA NAME', 'Crm Cd Desc', 'LAT', 'LON']
print("Missing values in key columns:")
print(df[key_cols].isnull().sum())

## 3. Temporal Analysis

In [ ]:
# Crime trends over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By year
yearly = df.groupby('Year').size()
axes[0, 0].bar(yearly.index, yearly.values, color='steelblue')
axes[0, 0].set_title('Crimes by Year')
axes[0, 0].set_xlabel('Year')

# By month
monthly = df.groupby('Month').size()
axes[0, 1].bar(monthly.index, monthly.values, color='coral')
axes[0, 1].set_title('Crimes by Month')
axes[0, 1].set_xlabel('Month')

# By day of week
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('DayOfWeek').size()
axes[1, 0].bar(dow_labels, daily.values, color='seagreen')
axes[1, 0].set_title('Crimes by Day of Week')

# By hour
hourly = df.groupby('Hour').size()
axes[1, 1].plot(hourly.index, hourly.values, 'purple', linewidth=2, marker='o')
axes[1, 1].set_title('Crimes by Hour of Day')
axes[1, 1].set_xlabel('Hour')
axes[1, 1].set_xlim(0, 23)

plt.tight_layout()
plt.show()

## 4. Crime Type Analysis

In [ ]:
# Top crime types
top_crimes = df['Crm Cd Desc'].value_counts().head(15)

plt.figure(figsize=(12, 8))
top_crimes.plot(kind='barh', color='indianred')
plt.title('Top 15 Crime Types')
plt.xlabel('Number of Incidents')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Crime heatmap: Hour vs Day of Week
heatmap_data = df.groupby(['DayOfWeek', 'Hour']).size().unstack(fill_value=0)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False)
plt.title('Crime Frequency: Day of Week vs Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.yticks([0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5], dow_labels, rotation=0)
plt.show()

## 5. Spatial Analysis

In [ ]:
# Crimes by area
area_counts = df['AREA NAME'].value_counts()

plt.figure(figsize=(12, 8))
area_counts.plot(kind='barh', color='teal')
plt.title('Crimes by Police District')
plt.xlabel('Number of Incidents')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Sample for mapping (full dataset is too large)
sample = df.dropna(subset=['LAT', 'LON']).sample(n=min(5000, len(df)), random_state=42)
sample = sample[(sample['LAT'] != 0) & (sample['LON'] != 0)]  # Remove zero coordinates

print(f"Mapping sample of {len(sample):,} incidents")

# Scatter plot of crime locations
plt.figure(figsize=(12, 10))
plt.scatter(sample['LON'], sample['LAT'], alpha=0.3, s=1, c='red')
plt.title('Crime Locations (Sample)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
%%ai $MODEL
Based on crime analysis patterns:

1. Why do crimes typically peak at certain hours (like noon and evening)?
2. What factors might explain higher crime rates in certain districts?
3. How could law enforcement use these temporal patterns for resource allocation?
4. What additional data would help predict crime hotspots?

## 6. Victim Demographics

In [ ]:
# Victim age distribution
victim_age = df['Vict Age'].dropna()
victim_age = victim_age[(victim_age > 0) & (victim_age < 120)]  # Clean outliers

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(victim_age, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Victim Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Victim sex
sex_counts = df['Vict Sex'].value_counts()
axes[1].pie(sex_counts.values, labels=sex_counts.index, autopct='%1.1f%%')
axes[1].set_title('Victim Sex Distribution')

plt.tight_layout()
plt.show()

## 7. Save Processed Summary

In [ ]:
# Create summary statistics
summary = {
    'total_crimes': len(df),
    'date_range': f"{df['DATE OCC'].min()} to {df['DATE OCC'].max()}",
    'top_crime': df['Crm Cd Desc'].mode().iloc[0],
    'most_affected_area': df['AREA NAME'].mode().iloc[0],
    'peak_hour': hourly.idxmax(),
    'avg_victim_age': victim_age.mean()
}

print("Dataset Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

---

## Summary

This notebook demonstrated:
- Loading and exploring large-scale crime data
- Temporal pattern analysis (yearly, monthly, hourly trends)
- Crime type distribution
- Spatial analysis by district
- Victim demographics analysis

**Next:** Try `Electric-Vehicle-Analysis.ipynb` for geospatial EV data

---